![image info](https://raw.githubusercontent.com/davidzarruk/MIAD_ML_NLP_2023/main/images/banner_1.png)

# Proyecto 1 - Predicción de popularidad en canción

En este proyecto podrán poner en práctica sus conocimientos sobre modelos predictivos basados en árboles y ensambles, y sobre la disponibilización de modelos. Para su desarrollo tengan en cuenta las instrucciones dadas en la "Guía del proyecto 1: Predicción de popularidad en canción".

**Entrega**: La entrega del proyecto deberán realizarla durante la semana 4. Sin embargo, es importante que avancen en la semana 3 en el modelado del problema y en parte del informe, tal y como se les indicó en la guía.

Para hacer la entrega, deberán adjuntar el informe autocontenido en PDF a la actividad de entrega del proyecto que encontrarán en la semana 4, y subir el archivo de predicciones a la competencia de Kaggle cuyo link estará disponible en la sección del Coursera del proyecto.

## Datos para la predicción de popularidad en cancion

En este proyecto se usará el conjunto de datos de datos de popularidad en canciones, donde cada observación representa una canción y se tienen variables como: duración de la canción, acusticidad y tempo, entre otras. El objetivo es predecir qué tan popular es la canción. Para más detalles puede visitar el siguiente enlace: [datos](https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset).

## Ejemplo predicción conjunto de test para envío a Kaggle

En esta sección encontrarán el formato en el que deben guardar los resultados de la predicción para que puedan subirlos a la competencia en Kaggle.

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Importación librerías
import pandas as pd
import numpy as np

In [3]:
# Carga de datos de archivo .csv
dataTraining = pd.read_csv('https://raw.githubusercontent.com/davidzarruk/MIAD_ML_NLP_2026/main/datasets/dataTrain_Spotify.csv')
dataTesting = pd.read_csv('https://raw.githubusercontent.com/davidzarruk/MIAD_ML_NLP_2026/main/datasets/dataTest_Spotify.csv', index_col=0)

In [4]:
# Visualización datos de entrenamiento
dataTraining.head()

,Unnamed: 0,track_id,artists,album_name,track_name,duration_ms,explicit,danceability,energy,key,...,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,popularity
0,0,7hUhmkALyQ8SX9mJs5XI3D,Love and Rockets,Love and Rockets,Motorcycle,211533,False,0.305,0.8490,9,...,1,0.0549,0.000058,0.056700,0.4640,0.3200,141.793,4,goth,22
1,1,5x59U89ZnjZXuNAAlc8X1u,Filippa Giordano,Filippa Giordano,"Addio del passato - From ""La traviata""",196000,False,0.287,0.1900,7,...,0,0.0370,0.930000,0.000356,0.0834,0.1330,83.685,4,opera,22
2,2,70Vng5jLzoJLmeLu3ayBQq,Susumu Yokota,Symbol,Purple Rose Minuet,216506,False,0.583,0.5090,1,...,1,0.0362,0.777000,0.202000,0.1150,0.5440,90.459,3,idm,37
3,3,1cRfzLJapgtwJ61xszs37b,Franz Liszt;YUNDI,Relajación y siestas,"Liebeslied (Widmung), S. 566",218346,False,0.163,0.0368,8,...,1,0.0472,0.991000,0.899000,0.1070,0.0387,69.442,3,classical,0
4,4,47d5lYjbiMy0EdMRV8lRou,Scooter,Scooter Forever,The Darkside,173160,False,0.647,0.9210,2,...,1,0.1850,0.000939,0.371000,0.1310,0.1710,137.981,4,techno,27


In [5]:
dataTesting.head()

,track_id,artists,album_name,track_name,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,6KwkVtXm8OUp2XffN5k7lY,Hillsong Worship,No Other Name,No Other Name,440247,False,0.369,0.598,7,-6.984,1,0.0304,0.00511,0.000000,0.176,0.0466,148.014,4,world-music
1,2dp5I5MJ8bQQHDoFaNRFtX,Internal Rot,Grieving Birth,Failed Organum,93933,False,0.171,0.997,7,-3.586,1,0.1180,0.00521,0.801000,0.420,0.0294,122.223,4,grindcore
2,5avw06usmFkFrPjX8NxC40,Zhoobin Askarieh;Ali Sasha,Noise A Noise 20.4-1,"Save the Trees, Pt. 1",213578,False,0.173,0.803,9,-10.071,0,0.1440,0.61300,0.001910,0.195,0.0887,75.564,3,iranian
3,75hT0hvlESnDJstem0JgyR,Bryan Adams,All I Want For Christmas Is You,Merry Christmas,151387,False,0.683,0.511,6,-5.598,1,0.0279,0.40600,0.000197,0.111,0.5980,109.991,3,rock
4,4bY2oZGA5Br3pTE1Jd1IfY,Nogizaka46,バレッタ TypeD,月の大きさ,236293,False,0.555,0.941,9,-3.294,0,0.0481,0.48400,0.000000,0.266,0.8130,92.487,4,j-idol


# Exploracion y Preprocesamiento de Datos

In [6]:
#TAMAÑO INICIAL
dataTraining.shape

(79800, 21)

In [7]:
dataTraining.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79800 entries, 0 to 79799
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        79800 non-null  int64  
 1   track_id          79800 non-null  object 
 2   artists           79800 non-null  object 
 3   album_name        79800 non-null  object 
 4   track_name        79800 non-null  object 
 5   duration_ms       79800 non-null  int64  
 6   explicit          79800 non-null  bool   
 7   danceability      79800 non-null  float64
 8   energy            79800 non-null  float64
 9   key               79800 non-null  int64  
 10  loudness          79800 non-null  float64
 11  mode              79800 non-null  int64  
 12  speechiness       79800 non-null  float64
 13  acousticness      79800 non-null  float64
 14  instrumentalness  79800 non-null  float64
 15  liveness          79800 non-null  float64
 16  valence           79800 non-null  float6

In [8]:
dataTraining.isnull().sum()

Unnamed: 0          0
track_id            0
artists             0
album_name          0
track_name          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
popularity          0
dtype: int64

Se confirma completitud de los datos, por lo que no hay que hacer preprocesamiento de datos nulos.

Ahora validaremos los valores unicos por variable:

In [9]:
for col in dataTraining.columns:
    print(f'Columna: {col} , Valores Unicos: {dataTraining[col].nunique()}, tipo: {dataTraining[col].dtype}')

Columna: Unnamed: 0 , Valores Unicos: 79800, tipo: int64
Columna: track_id , Valores Unicos: 66720, tipo: object
Columna: artists , Valores Unicos: 25775, tipo: object
Columna: album_name , Valores Unicos: 37315, tipo: object
Columna: track_name , Valores Unicos: 55767, tipo: object
Columna: duration_ms , Valores Unicos: 40712, tipo: int64
Columna: explicit , Valores Unicos: 2, tipo: bool
Columna: danceability , Valores Unicos: 1120, tipo: float64
Columna: energy , Valores Unicos: 1932, tipo: float64
Columna: key , Valores Unicos: 12, tipo: int64
Columna: loudness , Valores Unicos: 17562, tipo: float64
Columna: mode , Valores Unicos: 2, tipo: int64
Columna: speechiness , Valores Unicos: 1454, tipo: float64
Columna: acousticness , Valores Unicos: 4856, tipo: float64
Columna: instrumentalness , Valores Unicos: 5252, tipo: float64
Columna: liveness , Valores Unicos: 1706, tipo: float64
Columna: valence , Valores Unicos: 1737, tipo: float64
Columna: tempo , Valores Unicos: 37292, tipo: flo

Teniendo en cuenta lo anterior, se deciden eliminar columnas que representan identificadores unicos por no ser predictivos para los modelos a desarrollar, asi como los que presentan informacion relacionad a cada cancion:

In [10]:
columnas_a_quitar = ['Unnamed: 0','track_id','artists','album_name','track_name']
columnas_quitar_test = ['track_id','artists','album_name','track_name']
dataTraining = dataTraining.drop(columns=columnas_a_quitar)
dataTesting = dataTesting.drop(columns=columnas_quitar_test) #se deben quitar de los dos

In [11]:
print(dataTraining.shape)
print(dataTesting.shape)

(79800, 16)
(34200, 15)


Ahora procedemos a revisar la descripcion general de las variables del dataframe de *training*:

In [12]:
#division de numericas y categoricas
num_cols = dataTraining.select_dtypes(include=["int64", "float64", "bool"]).columns
cat_cols = dataTraining.select_dtypes(include=["object"]).columns

for col in cat_cols:
    dataTraining[col] = dataTraining[col].astype('category')

In [13]:
dataTraining.describe()

,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,popularity
count,7.980000e+04,79800.000000,79800.000000,79800.000000,79800.000000,79800.000000,79800.000000,79800.000000,79800.000000,79800.000000,79800.000000,79800.000000,79800.000000,79800.000000
mean,2.279022e+05,0.567318,0.641529,5.307043,-8.263741,0.637732,0.084750,0.314979,0.157319,0.213313,0.474267,122.076559,3.902556,33.265301
std,1.050599e+05,0.173110,0.251441,3.562186,5.035504,0.480659,0.105657,0.332512,0.310792,0.190075,0.259010,29.941937,0.434284,22.330871
min,1.338600e+04,0.000000,0.000019,0.000000,-49.307000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.744360e+05,0.456000,0.472000,2.000000,-10.006000,0.000000,0.035900,0.017000,0.000000,0.097900,0.261000,99.081250,4.000000,17.000000
50%,2.128130e+05,0.581000,0.685000,5.000000,-7.012000,1.000000,0.049000,0.169000,0.000041,0.132000,0.464000,122.009000,4.000000,35.000000
75%,2.614260e+05,0.695000,0.854000,8.000000,-5.000000,1.000000,0.084500,0.598000,0.050500,0.273000,0.684000,140.054000,4.000000,50.000000
max,5.237295e+06,0.985000,1.000000,11.000000,4.532000,1.000000,0.965000,0.996000,1.000000,1.000000,0.995000,222.605000,5.000000,100.000000


In [14]:
dataTraining.describe(include="category")

,track_genre
count,79800
unique,114
top,progressive-house
freq,738


In [15]:
#variables finales
dataTraining.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79800 entries, 0 to 79799
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   duration_ms       79800 non-null  int64   
 1   explicit          79800 non-null  bool    
 2   danceability      79800 non-null  float64 
 3   energy            79800 non-null  float64 
 4   key               79800 non-null  int64   
 5   loudness          79800 non-null  float64 
 6   mode              79800 non-null  int64   
 7   speechiness       79800 non-null  float64 
 8   acousticness      79800 non-null  float64 
 9   instrumentalness  79800 non-null  float64 
 10  liveness          79800 non-null  float64 
 11  valence           79800 non-null  float64 
 12  tempo             79800 non-null  float64 
 13  time_signature    79800 non-null  int64   
 14  track_genre       79800 non-null  category
 15  popularity        79800 non-null  int64   
dtypes: bool(1), category(1

Las variables: track_genre, key, mode y time_signature serán consideradas como *categoricas*, aunque varias son numericas en formato, al leer el dato en kaggle confirman el tipo (Adjunto ejemplo).

*key: La tonalidad en la que se encuentra la pista. Los enteros corresponden a notas usando la notación estándar de Clases de Tonos (por ejemplo, 0 = C, 1 = C♯/D♭, 2 = D, etc.). Si no se detecta tonalidad, el valor es -1.*

La variable explicit, aunque es bool, los modelos convierten ese false true en 0 y 1

**Division de los datos en Train y Test**

In [17]:
X = dataTraining.drop("popularity", axis=1)
y = dataTraining["popularity"]

In [18]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [19]:
print(X_train.shape)

(53466, 15)


In [20]:
X.columns.tolist()

['duration_ms',
 'explicit',
 'danceability',
 'energy',
 'key',
 'loudness',
 'mode',
 'speechiness',
 'acousticness',
 'instrumentalness',
 'liveness',
 'valence',
 'tempo',
 'time_signature',
 'track_genre']

Hacemos One Hot Encoding despues de la division de datos, solo en los datos de *Train*, para que el modelo no aprenda las categorias de los datos de test.

In [22]:
#nueva definicion de clasificacion de variables:
cat_cols = ["mode", "time_signature", "key", "track_genre"]
num_cols = ['duration_ms', 'explicit', 'danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

Para cumplir con el criterio: Preprocesamiento de datos (10 puntos)

*Los datos de entrenamiento se dividen en datos de entrenamiento y validación. Si decidieron preprocesar los datos (estandarizar, normalizar, imputar valores, etc), estos son correctamente preprocesados al ajustar sobre los datos de entrenamiento (.fit_transform()) y al transformar los datos del set de validación (.transform()).*

Se decide utilizar preprocessor, ya que se probo utilizar solo one hot encoding con las variables categoricas pero estaba generando errores al unir estas variables con las restantes.

In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols) #no les hace nada
    ]
)

#Ajustar y transformar:
X_train_p = preprocessor.fit_transform(X_train)
X_test_p = preprocessor.transform(X_test)

In [24]:
# Verificamos dimensiones
print(X_train_p.shape)
print(X_test_p.shape)

(53466, 144)
(26334, 144)


### Modelo 1: Decision Tree

Se inicia con un entrenamiento de arbol simple como base

In [22]:
from sklearn.tree import DecisionTreeRegressor
arbolrg = DecisionTreeRegressor(random_state=42)
arbolrg.fit(X_train_p, y_train)

DecisionTreeRegressor(random_state=42)

In [23]:
#evaluar modelo base en train y test
from sklearn.metrics import mean_squared_error

pred_train = arbolrg.predict(X_train_p)
pred_test = arbolrg.predict(X_test_p)
print('RSME Train: ',np.sqrt(mean_squared_error(y_train, pred_train)))
print('RSME Test: ',np.sqrt(mean_squared_error(y_test, pred_test)))

RSME Train:  3.4568646514192407
RSME Test:  22.43789204375564


El modelo base se equivoca en promedio 22 unidades en la prediccion de popularidad de una cancion con la base de test. Se incrementa mucho el error en test, lo cual evidencia overfitting, es conveniente calibrar hiperparametros que controlen el arbol

In [24]:
#se controla profundidad y se utiliza poda (Cost-Complexity Pruning)
# Rangos de hiperparámetros
max_depth_vals = range(1, 11) 
ccp_alpha_vals = np.linspace(0, 0.02, 21)
# Variables para guardar la mejor combinación
mejor_rmse = float("inf") #porque buscamos el minimo
mejor_depth = None
mejor_alpha = None
# Bucle pra probar las combinaciones de parametros
for depth in max_depth_vals:
    for alpha in ccp_alpha_vals:
        arbol_r = DecisionTreeRegressor(max_depth=depth, ccp_alpha=alpha, random_state=42)
        #se usa la base train creada previamente
        arbol_r.fit(X_train_p, y_train)
        #prediccion
        pred_test = arbol_r.predict(X_test_p)
        rmse = np.sqrt(mean_squared_error(y_test, pred_test))
        #comparacion
        if rmse < mejor_rmse:
            mejor_rmse = rmse
            mejor_depth = depth
            mejor_alpha = alpha

print(f"Mejor max_depth: {mejor_depth}")
print(f"Mejor ccp_alpha: {mejor_alpha}")
print(f"RMSE menor: {mejor_rmse}")

Mejor max_depth: 10
Mejor ccp_alpha: 0.017
RMSE menor: 21.092192934760096


### Modelo 2: XGBoost

In [26]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

xgbr = XGBRegressor(random_state=42)
#print(xgbr.get_params())
xgbr.fit(X_train_p, y_train)
#evaluar
pred_train = xgbr.predict(X_train_p)
pred_test = xgbr.predict(X_test_p)
print('RSME Train: ',np.sqrt(mean_squared_error(y_train, pred_train)))
print('RSME Test: ',np.sqrt(mean_squared_error(y_test, pred_test)))

RSME Train:  16.13204447324389
RSME Test:  18.202167692102122


No hay Overfitting, Ahora se inicia calibracion

#### Calibracion

Como la idea es buscar los mejores valores con early stopping, debo hacer nuevas divisiones (tener un set de validacion separado)

In [27]:
#nueva definicion de clasificacion de variables:
cat_cols = ["mode", "time_signature", "key", "track_genre"]
num_cols = ['duration_ms', 'explicit', 'danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

In [28]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols) #no les hace nada
    ]
)

In [29]:
# separar test final
X_total = dataTraining.drop("popularity", axis=1)
y_total = dataTraining["popularity"]

X_train_total, X_test, y_train_total, y_test = train_test_split(X_total, y_total, test_size=0.33, random_state=42)
# validacion
X_train, X_val, y_train, y_val = train_test_split(X_train_total, y_train_total, test_size=0.33, random_state=42)

In [30]:
#preprocessor
# fit SOLO con train
X_train_p = preprocessor.fit_transform(X_train)

# transformar los demás
X_val_p = preprocessor.transform(X_val)
X_test_p = preprocessor.transform(X_test)

In [57]:
# buscar el mejor n_estimators y learning rate, con early stopping
mejor_rmse = float("inf")
mejor_lr = None
mejor_n = None
lr_vals = np.linspace(0.01, 0.1, 5)
for lr in lr_vals:
    xgb2 = XGBRegressor(
        random_state=42,
        n_estimators=1000,
        learning_rate=lr,
        early_stopping_rounds=20,
        eval_metric="rmse"
    )
    #todo en validacion
    xgb2.fit(X_train_p, y_train, eval_set=[(X_val_p, y_val)], verbose=False)

    pred = xgb2.predict(X_val_p)
    rmse = np.sqrt(mean_squared_error(y_val, pred))   
    
    print(f"lr={lr}, n_estimator: {xgb2.best_iteration}, RMSE: {rmse}")

    if rmse < mejor_rmse:
        mejor_rmse = rmse
        mejor_lr = lr
        mejor_n = xgb2.best_iteration

print("\nMEJOR:")
print("lr:", mejor_lr)
print("n_estimators:", mejor_n)
print("RMSE:", mejor_rmse)
    
#evaluar
#pred_train = xgb2.predict(X_train_p)
#pred_test = xgb2.predict(X_test_p)
#print('RSME Train: ',np.sqrt(mean_squared_error(y_train, pred_train)))
#print('RSME Test: ',np.sqrt(mean_squared_error(y_test, pred_test)))

lr=0.01, n_estimator: 999, RMSE: 19.32439759809596
lr=0.0325, n_estimator: 999, RMSE: 18.424992447208425
lr=0.05500000000000001, n_estimator: 998, RMSE: 17.95250539482364
lr=0.0775, n_estimator: 999, RMSE: 17.67233434968871
lr=0.1, n_estimator: 999, RMSE: 17.511496990704956

MEJOR:
lr: 0.1
n_estimators: 999
RMSE: 17.511496990704956


Parece que hay margen de mejora, se profundizan parametros:

Se hacen varias iteraciones, en donde se ajustan parametros para validar si el LR mas bajo ayuda

In [64]:
# Complejidad del arbol
mx_dp_vals = np.arange(5, 12, 2)
min_ch_w_vals = [1,5] #np.arange(1, 6, 2)

for mxd in mx_dp_vals:
    for mcw in min_ch_w_vals:
        xgb3 = XGBRegressor(
            random_state=42,
            n_estimators=999,
            learning_rate=0.05, #se deja menor oara que aprenda mas
            max_depth=mxd,
            min_child_weight=mcw,
            early_stopping_rounds=20,
            eval_metric="rmse"
        )

        xgb3.fit(X_train_p, y_train, eval_set=[(X_val_p, y_val)], verbose=False)
        pred = xgb3.predict(X_vbest
        rmse = np.sqrt(mean_squared_error(y_val, pred))
        
        print(f"depth={mxd}, min child: {mcw}, n_estimator: {xgb3.best_iteration}, RMSE: {rmse}")      
        

depth=11, min child: 5, n_estimator: 1999, RMSE: 17.043237789390346


como veo que sigue mejorando pero no cambia n_estimator, ajusto LR

In [67]:
# Complejidad del arbol
mx_dp_vals = [11] # np.arange(5, 12, 2)
min_ch_w_vals = [5] #[1,5] #np.arange(1, 6, 2)

for mxd in mx_dp_vals:
    for mcw in min_ch_w_vals:
        xgb3 = XGBRegressor(
            random_state=42,
            n_estimators=4000, #balanceo entre lr y ne
            learning_rate=0.01, #se deja menor oara que aprenda mas
            max_depth=mxd,
            min_child_weight=mcw,
            early_stopping_rounds=20,
            eval_metric="rmse"
        )

        xgb3.fit(X_train_p, y_train, eval_set=[(X_val_p, y_val)], verbose=False)
        pred = xgb3.predict(X_val_p)
        rmse = np.sqrt(mean_squared_error(y_val, pred))
        
        print(f"depth={mxd}, min child: {mcw}, n_estimator: {xgb3.best_iteration}, RMSE: {rmse}")

depth=11, min child: 5, n_estimator: 3998, RMSE: 17.278798707248864


El ES no funciona, vamos a regularizar

In [92]:
#Regularizacion
ss = np.linspace(0.5, 1, 2)
cs = np.linspace(0.5, 1, 2)

for sample in ss:
    for csb in cs:
        xgb5 = XGBRegressor(
            random_state=42,
            n_estimators=3000,
            learning_rate=0.03,
            max_depth=11,
            min_child_weight=5,
            subsample=sample,
            colsample_bytree=csb,
            early_stopping_rounds=50, #ajuste
            eval_metric="rmse"
        )
        
        xgb5.fit(X_train_p, y_train, eval_set=[(X_val_p, y_val)], verbose=False)
        pred = xgb5.predict(X_val_p)
        rmse = np.sqrt(mean_squared_error(y_val, pred))
        
        print(f"n_estimator: {xgb5.best_iteration}, RMSE: {rmse}, ss: {sample}, cs {csb}")      

n_estimator: 1919, RMSE: 16.57452117550506, ss: 0.5, cs 0.5
n_estimator: 1678, RMSE: 16.784644893450047, ss: 0.5, cs 1.0
n_estimator: 2996, RMSE: 16.601736212326465, ss: 1.0, cs 0.5
n_estimator: 2666, RMSE: 16.9904390264215, ss: 1.0, cs 1.0


Mejoras muy pequeñas, se opta por cambio de enfoque

#### Importancia variables

In [70]:
#importancia de variables del mejor modelo
feature_names = preprocessor.get_feature_names_out()

importances = xbg4.feature_importances_

df_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

In [71]:
print(df_importance.head(20))

                             feature  importance
112         cat__track_genre_romance    0.078331
78          cat__track_genre_iranian    0.069839
100        cat__track_genre_pop-film    0.049848
43   cat__track_genre_detroit-techno    0.037491
32    cat__track_genre_chicago-house    0.029804
71       cat__track_genre_honky-tonk    0.028630
85             cat__track_genre_kids    0.028510
84            cat__track_genre_k-pop    0.027692
116       cat__track_genre_sertanejo    0.026338
73              cat__track_genre_idm    0.022592
127           cat__track_genre_tango    0.021388
113             cat__track_genre_sad    0.020610
35        cat__track_genre_classical    0.020601
34            cat__track_genre_chill    0.020249
24            cat__track_genre_anime    0.020231
86            cat__track_genre_latin    0.019348
61        cat__track_genre_grindcore    0.019036
74           cat__track_genre_indian    0.018506
29        cat__track_genre_breakbeat    0.015350
65            cat__t

In [73]:
print(df_importance.tail(20))

                             feature  importance
14                        cat__key_7    0.001116
130        cat__track_genre_trip-hop    0.001115
40        cat__track_genre_dancehall    0.001108
126       cat__track_genre_synth-pop    0.001105
141                    num__liveness    0.001100
4              cat__time_signature_3    0.001074
117      cat__track_genre_show-tunes    0.001056
11                        cat__key_4    0.001039
17                       cat__key_10    0.000972
1                        cat__mode_1    0.000948
119             cat__track_genre_ska    0.000948
0                        cat__mode_0    0.000947
94          cat__track_genre_new-age    0.000925
110     cat__track_genre_rock-n-roll    0.000878
92   cat__track_genre_minimal-techno    0.000835
6              cat__time_signature_5    0.000834
77       cat__track_genre_industrial    0.000821
36             cat__track_genre_club    0.000692
41      cat__track_genre_death-metal    0.000680
2              cat__

num__liveness no aporta, se quita junto con time_signature_0

In [32]:
#nueva definicion de clasificacion de variables:
cat_cols_ = ["mode", "key", "time_signature", "track_genre"]
num_cols_ = ['duration_ms', 'explicit', 'danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'valence', 'tempo']

In [33]:
cols_drop = [ "liveness","popularity"]  #"time_signature",
# separar test final
X_total_ = dataTraining.drop(cols_drop, axis=1)
y_total = dataTraining["popularity"]

X_train_total, X_test, y_train_total, y_test = train_test_split(X_total_, y_total, test_size=0.33, random_state=42)
# validacion
X_train, X_val, y_train, y_val = train_test_split(X_train_total, y_train_total, test_size=0.33, random_state=42)

In [34]:
#preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols_),
        ("num", "passthrough", num_cols_)
    ]
)
# fit SOLO con train
X_train_p = preprocessor.fit_transform(X_train)

# transformar los demás
X_val_p = preprocessor.transform(X_val)
X_test_p = preprocessor.transform(X_test)

In [36]:
#Uso del mejor modelo
xbg4 = XGBRegressor(
    random_state=42,
    n_estimators=3000,
    learning_rate=0.03,
    max_depth=11,
    min_child_weight=5,
    subsample=0.5,
    colsample_bytree=0.5,
    early_stopping_rounds=50, #ajuste
    eval_metric="rmse"
)

xbg4.fit(X_train_p, y_train, eval_set=[(X_val_p, y_val)], verbose=False)
pred = xbg4.predict(X_val_p)
rmse = np.sqrt(mean_squared_error(y_val, pred))

print(f"n_estimator: {xbg4.best_iteration}, RMSE: {rmse}")      

n_estimator: 1945, RMSE: 16.649740754363116


al quitar la variable, perdí información en conjunto. No fue superior.

Con la seleccion de hiperparametros importantes, voy a haecr CV

#### CV

Uso CV para: max_depth, learning_rate, subsample, etc.

Luego:
fijo n_estimators alto y uso de early_stopping_rounds

In [44]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [45]:
#nueva definicion de clasificacion de variables:
cat_cols = ["mode", "time_signature", "key", "track_genre"]
num_cols = ['duration_ms', 'explicit', 'danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

In [46]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols) #no les hace nada
    ]
)

#Ajustar y transformar:
#X_train_p = preprocessor.fit_transform(X_train)
#X_test_p = preprocessor.transform(X_test)

In [48]:
#GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

#se usa pipeline para que haga el fit_transform en cada fold
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBRegressor())
])

#Cuando usas Pipeline, el modelo ya no está “solo”, está dentro de un paso: deb dejar el nombre model__parametro
parametros = {
    'model__max_depth': [9, 11, 13],
    'model__learning_rate': [0.01, 0.03],
    'model__subsample': [0.5, 0.8], 
    'model__colsample_bytree': [0.5, 0.8]   
}

#xgbcv = XGBRegressor()

grid = GridSearchCV(
    pipeline,
    parametros,
    cv=5,              
    scoring='neg_root_mean_squared_error'
)

grid.fit(X_train, y_train) #sin transformar

print(grid.best_params_)
print(-grid.best_score_)

{'model__colsample_bytree': 0.8, 'model__learning_rate': 0.03, 'model__max_depth': 13, 'model__subsample': 0.5}
-19.25140838623047


In [50]:
#modelo con ES: Crear el conjunto de validacion
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split( X_train, y_train, test_size=0.33, random_state=42)

In [51]:
# Usar el preprocessor 
X_tr_p = preprocessor.fit_transform(X_tr)
X_val_p = preprocessor.transform(X_val)
X_test_p = preprocessor.transform(X_test)

In [54]:
#limpiar nombres de parametros para usarlos 
best_params= grid.best_params_
best_params_limpio = {
    k.replace("model__", ""): v
    for k, v in best_params.items()
}

In [55]:
#Uso del mejor modelo
xgb6 = XGBRegressor(
    random_state=42,
    **best_params_limpio,
    n_estimators=3000,
    min_child_weight=5,
    early_stopping_rounds=50, #ajuste
    eval_metric="rmse"
)

xgb6.fit(X_tr_p, y_tr, eval_set=[(X_val_p, y_val)], verbose=False)
pred = xgb6.predict(X_val_p)
rmse = np.sqrt(mean_squared_error(y_val, pred))

print(f"n_estimator: {xgb6.best_iteration}, RMSE: {rmse}")    

n_estimator: 1391, RMSE: 16.53498229961065


In [63]:
#comparar train y val para overfitt
#Uso del mejor modelo
xgb6 = XGBRegressor(
    random_state=42,
    **best_params_limpio,
    n_estimators=3000,
    min_child_weight=5,
    early_stopping_rounds=50, #ajuste
    eval_metric="rmse"
)

xgb6.fit(X_tr_p, y_tr, eval_set=[(X_tr_p, y_tr),(X_val_p, y_val)], verbose=100)  #verbose - avances 

[0]	validation_0-rmse:22.26729	validation_1-rmse:22.27096
[100]	validation_0-rmse:18.39405	validation_1-rmse:19.30410
[200]	validation_0-rmse:16.56338	validation_1-rmse:18.43747
[300]	validation_0-rmse:14.96258	validation_1-rmse:17.88386
[400]	validation_0-rmse:13.51758	validation_1-rmse:17.49852
[500]	validation_0-rmse:12.11558	validation_1-rmse:17.19096
[600]	validation_0-rmse:10.85513	validation_1-rmse:16.98405
[700]	validation_0-rmse:9.79798	validation_1-rmse:16.82554
[800]	validation_0-rmse:8.99023	validation_1-rmse:16.74731
[900]	validation_0-rmse:8.24316	validation_1-rmse:16.67448
[1000]	validation_0-rmse:7.63305	validation_1-rmse:16.63042
[1100]	validation_0-rmse:7.07569	validation_1-rmse:16.58871
[1200]	validation_0-rmse:6.63004	validation_1-rmse:16.56054
[1300]	validation_0-rmse:6.22173	validation_1-rmse:16.54304
[1400]	validation_0-rmse:5.91911	validation_1-rmse:16.53741
[1441]	validation_0-rmse:5.81498	validation_1-rmse:16.53804


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=50,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.03, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=13,
             max_leaves=None, min_child_weight=5, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=3000,
             n_jobs=None, num_parallel_tree=None, ...)

Hay ligero overfitting, Aprende patrones del train muy bien, buen rmse pero esos patrones no generalizan completamente

Si train RMSE << val RMSE → baja max_depth

In [ ]:
parametros = {
    'max_depth': [5, 7, 9, 11],  # más realista
}

# Generar archivo

Cuando tengo el mejor combinacion, debo re-entrenar con todos los datos de training

In [56]:
X_full = dataTraining.drop("popularity", axis=1)
y_full = dataTraining["popularity"]

In [57]:
#preprocessor
X_full_p = preprocessor.fit_transform(X_full)
X_test_kaggle = dataTesting.copy()
X_test_kaggle_p = preprocessor.transform(X_test_kaggle)

In [58]:
#entrenar modelo seleccionado: 16.53498229961065
modelo_final = XGBRegressor(
    random_state=42,
    n_estimators=3000,
    learning_rate=0.03,
    max_depth=13,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.5,
    eval_metric="rmse"
)

modelo_final.fit(X_full_p, y_full)

preds = modelo_final.predict(X_test_kaggle_p)
preds = preds.clip(0, 100)

In [59]:
submission = pd.DataFrame({
    "ID": dataTesting.index,
    "Popularity": preds
})

submission.to_csv("submission_2.csv", index=False)

In [107]:
len(dataTesting)

34200

In [61]:
import pandas as pd

submission = pd.read_csv("submission_2.csv")

# 1. Columnas correctas
print("Columnas:", submission.columns.tolist())

# 2. Tamaño
print("Filas:", len(submission))
print(submission.dtypes)

# 3. Ver primeras filas
print(submission.head())

# 4. Ver últimas filas
print(submission.tail())

# 5. Revisar nulos
print("Nulos:", submission.isnull().sum())

# 6. Revisar rango
print("Min:", submission["Popularity"].min())
print("Max:", submission["Popularity"].max())

Columnas: ['ID', 'Popularity']
Filas: 34200
ID              int64
Popularity    float64
dtype: object
   ID  Popularity
0   0   46.020140
1   1   15.889379
2   2    1.087871
3   3    0.178422
4   4   22.669160
          ID  Popularity
34195  34195   27.031506
34196  34196   43.564290
34197  34197   55.465683
34198  34198   29.390488
34199  34199   44.458954
Nulos: ID            0
Popularity    0
dtype: int64
Min: 0.0
Max: 86.90636
